In [24]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

# Load the cleaned dataframe exported from credit_risk_investigation.ipynb
df = pd.read_csv("data/credit_risk_dataset_cleaned.csv")
df.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file
0,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N
1,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N
2,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N
3,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y
4,21,9900,OWN,2.0,VENTURE,A,2500,7.14,1,0.25,N


# Logistic Regression Baseline

This notebook trains and evaluates an interpretable logistic-regression baseline for binary loan-default prediction. The cleaned dataset is produced by `credit_risk_investigation.ipynb`.

`X` contains applicant and loan information; `y` is `loan_status`, where 1 indicates default and 0 indicates repayment.

In [25]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

X = df.drop("loan_status", axis=1)
y = df["loan_status"]

print("Features:", X.shape)
print("Target:", y.shape)
print(f"Default rate: {y.mean():.1%}")

Features: (31673, 10)
Target: (31673,)
Default rate: 21.9%


In [26]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Training default rate: {y_train.mean():.1%}")
print(f"Test default rate: {y_test.mean():.1%}")

Training set: (25338, 10)
Test set: (6335, 10)
Training default rate: 21.9%
Test default rate: 21.9%


The dataset has an imbalanced target, so `stratify=y` preserves the default rate across the train/test split.

## Preprocessing

In [28]:
categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

numerical_features = X.select_dtypes(
    exclude=["object"]
).columns.tolist()

print("Categorical features:")
print(categorical_features)

print("\nNumerical features:")
print(numerical_features)

Categorical features:
['person_home_ownership', 'loan_intent', 'loan_grade', 'cb_person_default_on_file']

Numerical features:
['person_age', 'person_income', 'person_emp_length', 'loan_amnt', 'loan_int_rate', 'loan_percent_income']


/var/folders/b2/5jwynvcd32x0z9pz7rvkd9yr0000gn/T/ipykernel_1853/1838352015.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(


In [29]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numerical_features
        ),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

# Logistic Regression 

This will be the baseline model. Logistic regression was used as it is interpretable and relatively simple, appropriate for binary classification.
Note the `class_weight="balanced"` used to give greater weight to the minority class as there was an imbalance.

In [30]:
from sklearn.linear_model import LogisticRegression

logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)

In [31]:
# Train the model
logistic_model.fit(X_train, y_train)
logistic_predictions = logistic_model.predict(X_test)
logistic_probabilities = logistic_model.predict_proba(X_test)[:, 1]

`predict` returns binary class labels, while `predict_proba` returns the estimated probability of default.

For credit risk, probabilities are useful because a lender may want a risk estimate rather than only a default/no-default decision.

## Evaluation

In [32]:
metrics = pd.DataFrame(
    {
        "metric": ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"],
        "value": [
            accuracy_score(y_test, logistic_predictions),
            precision_score(y_test, logistic_predictions, zero_division=0),
            recall_score(y_test, logistic_predictions, zero_division=0),
            f1_score(y_test, logistic_predictions, zero_division=0),
            roc_auc_score(y_test, logistic_probabilities),
        ],
    }
).set_index("metric")

print(metrics.round(3))
print("\nConfusion matrix [rows = actual, columns = predicted]:")
print(confusion_matrix(y_test, logistic_predictions))

           value
metric          
Accuracy   0.812
Precision  0.550
Recall     0.774
F1         0.644
ROC-AUC    0.873

Confusion matrix [rows = actual, columns = predicted]:
[[4069  878]
 [ 313 1075]]


In [33]:
print(classification_report(y_test, logistic_predictions))

              precision    recall  f1-score   support

           0       0.93      0.82      0.87      4947
           1       0.55      0.77      0.64      1388

    accuracy                           0.81      6335
   macro avg       0.74      0.80      0.76      6335
weighted avg       0.85      0.81      0.82      6335



Accuracy provides an overall measure of classification performance, but recall is particularly relevant to credit-risk modelling because failing to identify a borrower who subsequently defaults may result in financial losses.

Recall is not always the most important metric. Instead, metric selection depends on business costs.
